# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# Install
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
# Imports
import os
import json
import math
import random
import zipfile
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import (AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig,)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training,)
from transformers import get_linear_schedule_with_warmup

# Configuration
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
ZIP_PATH = "/content/pixels-to-predictions.zip"
EXTRACT_DIR = Path("/content/pixels_data")

CHOICE_LETTERS = "ABCDEFGHIJ"

RANDOM_STATE = 42
BATCH_SIZE = 2
LR = 1e-4
MAX_TRAIN_STEPS = 500

# Current best experimental setup from our Colab tuning
TRAIN_SUBSET_SIZE = 2500
VAL_SUBSET_SIZE = 400

device = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Device:", device)

Device: cuda


## 2. Load and Preprocess Data

In [ ]:
# Unzip data
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Unzipped to:", EXTRACT_DIR)
print("Contents:", os.listdir(EXTRACT_DIR))

Unzipped to: /content/pixels_data
Contents: ['sample_submission.csv', 'val.csv', 'test.csv', 'images', 'train.csv']


In [ ]:
# Path
DATA_DIR = EXTRACT_DIR
IMG_ROOT = DATA_DIR / "images"

print("DATA_DIR:", DATA_DIR)
print("IMG_ROOT:", IMG_ROOT)
print("IMG_ROOT exists:", IMG_ROOT.exists())

# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# parse choices column
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(train_df.columns.tolist())
train_df.head(2)

DATA_DIR: /content/pixels_data
IMG_ROOT: /content/pixels_data/images
IMG_ROOT exists: True
Train: 3,109 | Val: 1,048 | Test: 1,008
['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [ ]:
# Prompt Engineering
def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())

    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"

    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Select the correct answer (A, B, C, D, ...). Respond with ONLY the letter.\n"
    prompt += "Answer:"
    # prompt += "The correct answer is:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

print(build_prompt(train_df.iloc[0], include_answer=True))


<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [ ]:
'''
def build_prompt_alt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())

    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"

    prompt += f"Read the question and choose the best option.\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Options:\n{choices_str}\n"
    prompt += "Select one option. Output only the letter.\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt
    '''

In [ ]:
'''
def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())

    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    meta_parts = []
    for col in ["subject", "topic", "category", "grade"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            meta_parts.append(f"{col}: {row[col]}")

    metadata_str = " | ".join(meta_parts)

    prompt = "<image>\n"

    if metadata_str:
        prompt += f"Metadata: {metadata_str}\n\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"

    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt
'''

'\ndef build_prompt(row: pd.Series, include_answer: bool = False) -> str:\n    context_parts = []\n\n    lecture = row.get("lecture", "")\n    hint = row.get("hint", "")\n\n    if pd.notna(lecture) and str(lecture).strip():\n        context_parts.append(str(lecture).strip())\n\n    if pd.notna(hint) and str(hint).strip():\n        context_parts.append(str(hint).strip())\n\n    context_str = "\n".join(context_parts)\n\n    choices = row["choices"]\n    choices_str = "\n".join(\n        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)\n    )\n\n    meta_parts = []\n    for col in ["subject", "topic", "category", "grade"]:\n        if col in row and pd.notna(row[col]) and str(row[col]).strip():\n            meta_parts.append(f"{col}: {row[col]}")\n\n    metadata_str = " | ".join(meta_parts)\n\n    prompt = "<image>\n"\n\n    if metadata_str:\n        prompt += f"Metadata: {metadata_str}\n\n"\n\n    if context_str:\n        prompt += f"Context:\n{context_str}\n\n"\n\n    prompt

In [ ]:
# PyTorch
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_root: Path, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.img_root = Path(img_root)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img_path = self.img_root / rel_path
        img = Image.open(img_path).convert("RGB")
        return img

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        return {
            "image": img,
            "text": build_prompt(row, include_answer=self.is_train),
            "answer": int(row["answer"]) if "answer" in row and pd.notna(row["answer"]) else -1,
            "choices": row["choices"],
            "id": row["id"] if "id" in row else idx,
        }

In [ ]:
# Check dataset output
train_ds_full = ScienceQADataset(train_df, IMG_ROOT, is_train=True)
sample = train_ds_full[0]

print("ID:", sample["id"])
print("Answer:", sample["answer"])
print("Choices:", sample["choices"])
print("Image size:", sample["image"].size)
print(sample["text"][:700])

ID: train_07667
Answer: 2
Choices: ["the male's tadpoles will be larger when they hatch", 'the male will carry his tadpoles through the forest', "the male's tadpoles will become adult frogs"]
Image size: (302, 252)
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will surv


In [ ]:
# Load Processor and 4-bit SmolVLM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


In [ ]:
# Inspect Linear Module Name
linear_names = set()

for name, module in model.named_modules():
    if "Linear" in str(type(module)):
        linear_names.add(name.split(".")[-1])

print(sorted(linear_names))

['down_proj', 'fc1', 'fc2', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'out_proj', 'proj', 'q_proj', 'up_proj', 'v_proj']


In [ ]:
# LoRA Adapter
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    # lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    # target_modules = ["q_proj", "v_proj", "up_proj", "down_proj", "gate_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # use_dora=True,
)

model = get_peft_model(model, lora_config)

In [ ]:
# Parameter count check
def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total
    return trainable, total, pct

trainable, total, pct = count_trainable_params(model)
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")
print(f"Trainable %:      {pct:.4f}%")

assert trainable <= 5_000_000, "Trainable parameters exceed competition limit!"

Trainable params: 3,883,008
Total params:     305,713,344
Trainable %:      1.2701%


In [ ]:
# Collate
def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"] for x in batch]

    model_inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    input_ids = model_inputs["input_ids"]
    labels = torch.full_like(input_ids, -100)

    num_missing = 0
    missing_examples = []

    for i, item in enumerate(batch):
        answer_idx = item["answer"]
        if answer_idx < 0:
            continue

        answer_letter = CHOICE_LETTERS[answer_idx]
        candidate_texts = [
            " " + answer_letter,
            answer_letter,
            "\n" + answer_letter,
            ": " + answer_letter,
        ]

        found = False
        seq = input_ids[i].tolist()

        for cand in candidate_texts:
            cand_ids = processor.tokenizer(
                cand,
                add_special_tokens=False,
                return_tensors="pt"
            )["input_ids"][0].tolist()

            for start in range(len(seq) - len(cand_ids), -1, -1):
                if seq[start:start+len(cand_ids)] == cand_ids:
                    labels[i, start:start+len(cand_ids)] = input_ids[i, start:start+len(cand_ids)]
                    found = True
                    break

            if found:
                break

        if not found:
            num_missing += 1
            missing_examples.append({
                "id": item["id"],
                "answer_idx": answer_idx,
                "answer_letter": answer_letter,
                "text_tail": item["text"][-120:]
            })

    labels[input_ids == processor.tokenizer.pad_token_id] = -100

    model_inputs["labels"] = labels
    model_inputs["answers"] = [x["answer"] for x in batch]
    model_inputs["ids"] = [x["id"] for x in batch]
    model_inputs["choices"] = [x["choices"] for x in batch]
    model_inputs["num_missing_labels"] = num_missing
    model_inputs["missing_examples"] = missing_examples

    return model_inputs

In [ ]:
# Debug check
debug_loader = DataLoader(
    train_ds_full,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

debug_batch = next(iter(debug_loader))

print("num_missing_labels:", debug_batch["num_missing_labels"])
print("missing_examples:", debug_batch["missing_examples"][:2])

for i in range(len(debug_batch["ids"])):
    kept = debug_batch["labels"][i] != -100
    print("\nID:", debug_batch["ids"][i])
    print("Answer idx:", debug_batch["answers"][i])
    print("Decoded kept tokens:", processor.tokenizer.decode(debug_batch["input_ids"][i][kept]))

num_missing_labels: 0
missing_examples: []

ID: train_04398
Answer idx: 2
Decoded kept tokens:  C

ID: train_12255
Answer idx: 1
Decoded kept tokens:  B

ID: train_00813
Answer idx: 0
Decoded kept tokens:  A

ID: train_10663
Answer idx: 1
Decoded kept tokens:  B


In [ ]:
# Create subset
# train_df_small = train_df.sample(TRAIN_SUBSET_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
# val_df_small   = val_df.sample(VAL_SUBSET_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

# train_df_small = train_df.copy().reset_index(drop=True)
# val_df_small   = val_df.sample(400, random_state=RANDOM_STATE).reset_index(drop=True)

HARD_RATIO = 0.30

HARD_CATEGORIES = [
    "Velocity, acceleration, and forces",
    "Particle motion and energy",
    "Magnets",
    "States of matter",
    "Genes to traits",
]

SECONDARY_CATEGORIES = [
    "Ecological interactions",
    "Solutions",
    "Weather and climate",
]

base_df = train_df.copy()

hard_df = base_df[base_df["category"].isin(HARD_CATEGORIES)].copy()
secondary_df = base_df[base_df["category"].isin(SECONDARY_CATEGORIES)].copy()

hard_idx = set(hard_df.index) | set(secondary_df.index)
regular_df = base_df.drop(index=list(hard_idx)).copy()

n_hard = int(TRAIN_SUBSET_SIZE * 0.20)
n_secondary = int(TRAIN_SUBSET_SIZE * 0.10)
n_regular = TRAIN_SUBSET_SIZE - n_hard - n_secondary

train_regular = regular_df.sample(
    n_regular,
    random_state=RANDOM_STATE,
    replace=len(regular_df) < n_regular
)

train_hard = hard_df.sample(
    n_hard,
    random_state=RANDOM_STATE,
    replace=len(hard_df) < n_hard
)

train_secondary = secondary_df.sample(
    n_secondary,
    random_state=RANDOM_STATE,
    replace=len(secondary_df) < n_secondary
)

train_df_small = pd.concat(
    [train_regular, train_hard, train_secondary]
).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

val_df_small = val_df.sample(400, random_state=RANDOM_STATE).reset_index(drop=True)

print("Train subset:", len(train_df_small))
print("Val subset:", len(val_df_small))
print("Physics-hard examples:", len(train_hard))
print("Secondary-hard examples:", len(train_secondary))

print("\nTop categories:")
print(train_df_small["category"].value_counts().head(20))

print("\nnum_choices distribution:")
print(train_df_small["num_choices"].value_counts(normalize=True))


Train subset: 2500
Val subset: 400
Physics-hard examples: 500
Secondary-hard examples: 250

Top categories:
category
Designing experiments                  204
Ecosystems                             162
Maps                                   160
Engineering practices                  150
Solutions                              147
Basic economic principles              144
Geography                              141
Velocity, acceleration, and forces     132
Particle motion and energy             119
Magnets                                119
Classification and scientific names    102
Adaptations                             89
Genes to traits                         87
Scientific names                        83
English colonies in North America       76
Colonial America                        64
Ecological interactions                 58
Classification                          48
Atoms and molecules                     46
Weather and climate                     45
Name: count, dtype: int

In [ ]:
# Build dataset
train_ds = ScienceQADataset(train_df_small, IMG_ROOT, is_train=True)
val_ds   = ScienceQADataset(val_df_small,   IMG_ROOT, is_train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

print("Train examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train examples: 2500
Validation examples: 400
Train batches: 1250
Val batches: 200


In [ ]:
# Training
optimizer = AdamW(model.parameters(), lr=LR)
# optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

model.train()
running_losses = []

num_training_steps = MAX_TRAIN_STEPS
num_warmup_steps = int(0.05 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

model.train()
running_losses = []

for step, batch in enumerate(tqdm(train_loader, total=MAX_TRAIN_STEPS)):
    batch = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in batch.items()
        if k in ["input_ids", "attention_mask", "pixel_values", "pixel_attention_mask", "labels"]
    }

    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    running_losses.append(loss.item())

    if (step + 1) % 10 == 0:
        avg_loss = sum(running_losses[-10:]) / 10
        print(f"step={step+1}, avg_loss(last 10)={avg_loss:.4f}")

    if step + 1 >= MAX_TRAIN_STEPS:
        break

  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


step=10, avg_loss(last 10)=0.9105
step=20, avg_loss(last 10)=1.2291
step=30, avg_loss(last 10)=0.7502
step=40, avg_loss(last 10)=1.1392
step=50, avg_loss(last 10)=0.7205
step=60, avg_loss(last 10)=1.1782
step=70, avg_loss(last 10)=0.6321
step=80, avg_loss(last 10)=0.8608
step=90, avg_loss(last 10)=0.8904
step=100, avg_loss(last 10)=0.6768
step=110, avg_loss(last 10)=0.8515
step=120, avg_loss(last 10)=1.5463
step=130, avg_loss(last 10)=1.0775
step=140, avg_loss(last 10)=0.5605
step=150, avg_loss(last 10)=0.7307
step=160, avg_loss(last 10)=0.5541
step=170, avg_loss(last 10)=0.8606
step=180, avg_loss(last 10)=1.2326
step=190, avg_loss(last 10)=0.9014
step=200, avg_loss(last 10)=1.2387
step=210, avg_loss(last 10)=1.3041
step=220, avg_loss(last 10)=1.3401
step=230, avg_loss(last 10)=0.6048
step=240, avg_loss(last 10)=1.0278
step=250, avg_loss(last 10)=0.3936
step=260, avg_loss(last 10)=0.3107
step=270, avg_loss(last 10)=1.7328
step=280, avg_loss(last 10)=0.3908
step=290, avg_loss(last 10)=0

In [ ]:
#  Scoring evaluation
def score_choices_with_letters(model, processor, row, img_root):
    model.eval()

    image = Image.open(img_root / row["image_path"]).convert("RGB")
    prompt = build_prompt(row, include_answer=False)

    base_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    base_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in base_inputs.items()
    }

    choice_scores = []
    num_choices = len(row["choices"])

    for choice_idx in range(num_choices):
        letter = CHOICE_LETTERS[choice_idx]

        answer_text = " " + letter
        answer_ids = processor.tokenizer(
            answer_text,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"].to(model.device)

        input_ids = torch.cat([base_inputs["input_ids"], answer_ids], dim=1)
        attention_mask = torch.cat([
            base_inputs["attention_mask"],
            torch.ones_like(answer_ids, device=model.device)
        ], dim=1)

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=base_inputs["pixel_values"],
                pixel_attention_mask=base_inputs.get("pixel_attention_mask", None),
            )

        logits = outputs.logits[:, :-1, :]
        target_ids = input_ids[:, 1:]

        answer_len = answer_ids.shape[1]
        answer_logits = logits[:, -answer_len:, :]
        answer_targets = target_ids[:, -answer_len:]

        log_probs = F.log_softmax(answer_logits, dim=-1)
        token_log_probs = log_probs.gather(-1, answer_targets.unsqueeze(-1)).squeeze(-1)

        score = token_log_probs.mean().item()
        choice_scores.append(score)

    pred = int(torch.tensor(choice_scores).argmax().item())
    return pred, choice_scores


def evaluate_with_scoring(model, processor, df, img_root, max_samples=200):
    preds = []
    refs = []

    subset = df.iloc[:max_samples].reset_index(drop=True)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        pred, _ = score_choices_with_letters(model, processor, row, img_root)
        preds.append(pred)
        refs.append(int(row["answer"]))

    acc = sum(int(p == r) for p, r in zip(preds, refs)) / len(refs)
    return acc, preds, refs

In [ ]:
'''
def score_choices_with_prompt(model, processor, row, img_root, prompt_fn):
    model.eval()

    image = Image.open(img_root / row["image_path"]).convert("RGB")
    prompt = prompt_fn(row, include_answer=False)

    base_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    base_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in base_inputs.items()
    }

    choice_scores = []
    num_choices = len(row["choices"])

    for choice_idx in range(num_choices):
        letter = CHOICE_LETTERS[choice_idx]

        answer_text = " " + letter
        answer_ids = processor.tokenizer(
            answer_text,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"].to(model.device)

        input_ids = torch.cat([base_inputs["input_ids"], answer_ids], dim=1)
        attention_mask = torch.cat([
            base_inputs["attention_mask"],
            torch.ones_like(answer_ids, device=model.device)
        ], dim=1)

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=base_inputs["pixel_values"],
                pixel_attention_mask=base_inputs.get("pixel_attention_mask", None),
            )

        logits = outputs.logits[:, :-1, :]
        target_ids = input_ids[:, 1:]

        answer_len = answer_ids.shape[1]
        answer_logits = logits[:, -answer_len:, :]
        answer_targets = target_ids[:, -answer_len:]

        log_probs = F.log_softmax(answer_logits, dim=-1)
        token_log_probs = log_probs.gather(
            -1,
            answer_targets.unsqueeze(-1)
        ).squeeze(-1)

        score = token_log_probs.mean().item()
        choice_scores.append(score)

    pred = int(np.argmax(choice_scores))
    return pred, choice_scores
    '''

In [ ]:
'''
def score_choices_dual_prompt(model, processor, row, img_root):
    _, scores_a = score_choices_with_prompt(
        model=model,
        processor=processor,
        row=row,
        img_root=img_root,
        prompt_fn=build_prompt
    )

    _, scores_b = score_choices_with_prompt(
        model=model,
        processor=processor,
        row=row,
        img_root=img_root,
        prompt_fn=build_prompt_alt
    )

    scores_a = np.array(scores_a)
    scores_b = np.array(scores_b)

    avg_scores = (scores_a + scores_b) / 2.0
    pred = int(np.argmax(avg_scores))

    return pred, avg_scores
    '''

In [ ]:
'''
def evaluate_with_dual_prompt(model, processor, df, img_root, max_samples=400):
    preds = []
    refs = []

    subset = df.iloc[:max_samples].reset_index(drop=True)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        pred, _ = score_choices_dual_prompt(model, processor, row, img_root)
        preds.append(pred)
        refs.append(int(row["answer"]))

    acc = sum(int(p == r) for p, r in zip(preds, refs)) / len(refs)
    return acc, preds, refs
    '''

In [ ]:
'''
dual_acc, dual_preds, dual_refs = evaluate_with_dual_prompt(
    model=model,
    processor=processor,
    df=val_df_small,
    img_root=IMG_ROOT,
    max_samples=400
)

print("Dual-prompt validation accuracy:", dual_acc)
'''

  0%|          | 0/400 [00:00<?, ?it/s]

Dual-prompt validation accuracy: 0.6625


In [ ]:
# Validation
val_acc_score, val_preds_score, val_refs_score = evaluate_with_scoring(
    model=model,
    processor=processor,
    df=val_df_small,
    img_root=IMG_ROOT,
    max_samples=400
)

print("Validation accuracy (first 400 samples):", val_acc_score)

  0%|          | 0/400 [00:00<?, ?it/s]

Validation accuracy (first 400 samples): 0.6725


In [ ]:
# Error analysis dataframe
n_eval = len(val_preds_score)

error_df = val_df_small.iloc[:n_eval].copy()
error_df["pred"] = val_preds_score
error_df["ref"] = val_refs_score
error_df["correct"] = error_df["pred"] == error_df["ref"]

print("Overall accuracy:", error_df["correct"].mean())
error_df.head()

Overall accuracy: 0.6775


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill,pred,ref,correct
0,val_00570,images/val/val_00570.png,"Based on the text, what is one thing that spin...","[They spin around in the air., They hunt for f...",3,0,Read the text about spinner dolphins.\nHave yo...,NaN,Look at the text in bold below. It tells you t...,closed choice,grade3,language science,reading-comprehension,Informational texts: level 1,Read passages about animals,1,0,False
1,val_03508,images/val/val_03508.png,Which solution has a higher concentration of y...,"[neither; their concentrations are the same, S...",3,2,The diagram below is a model of two solutions....,A solution is made up of two or more substance...,"In Solution A and Solution B, the yellow parti...",closed choice,grade7,natural science,chemistry,Solutions,Compare concentrations of solutions,2,2,True
2,val_03587,images/val/val_03587.png,Which solution has a higher concentration of p...,"[Solution B, Solution A, neither; their concen...",3,0,The diagram below is a model of two solutions....,A solution is made up of two or more substance...,"In Solution A and Solution B, the pink particl...",closed choice,grade7,natural science,chemistry,Solutions,Compare concentrations of solutions,0,0,True
3,val_00603,images/val/val_00603.png,Which of these cities is marked on the map?,"[San Diego, San Francisco, Las Vegas, Salt Lak...",4,3,NaN,NaN,"The city is Salt Lake City, Utah. San Francisc...",closed choice,grade4,social science,geography,Cities,Cities of the West,3,3,True
4,val_02288,images/val/val_02288.png,Complete the text to describe the diagram.\nSo...,"[to the right than to the left, to the left th...",2,1,The diagram below shows a solution with one so...,"In a solution, solute particles move and sprea...",Look at the diagram again. It shows you how th...,closed choice,grade6,natural science,chemistry,Solutions,Diffusion across membranes,0,1,False


In [ ]:
print("Accuracy by number of choices:")
display(
    error_df.groupby("num_choices")["correct"]
    .agg(["count", "mean"])
    .sort_index()
)

Accuracy by number of choices:


,count,mean
num_choices,,
2,99,0.757576
3,199,0.623116
4,87,0.770115
5,15,0.333333


In [ ]:
print("Accuracy by subject:")
display(
    error_df.groupby("subject")["correct"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

print("Accuracy by grade:")
display(
    error_df.groupby("grade")["correct"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

Accuracy by subject:


,count,mean
subject,,
language science,8,0.375000
natural science,305,0.642623
social science,87,0.827586


Accuracy by grade:


,count,mean
grade,,
grade12,2,0.500000
grade8,96,0.625000
grade3,52,0.634615
grade5,39,0.641026
grade6,75,0.666667
grade7,63,0.682540
grade2,9,0.777778
grade4,64,0.812500


In [ ]:
print("Worst categories:")
display(
    error_df.groupby("category")["correct"]
    .agg(["count", "mean"])
    .query("count >= 5")
    .sort_values("mean")
    .head(15)
)

print("Worst topics:")
display(
    error_df.groupby("topic")["correct"]
    .agg(["count", "mean"])
    .query("count >= 5")
    .sort_values("mean")
    .head(15)
)

Worst categories:


,count,mean
category,,
Genes to traits,16,0.062500
Solutions,24,0.333333
Ecological interactions,10,0.400000
"Velocity, acceleration, and forces",18,0.444444
Particle motion and energy,26,0.461538
Magnets,30,0.566667
Persuasive strategies,5,0.600000
States of matter,10,0.600000
Astronomy,16,0.687500


Worst topics:


,count,mean
topic,,
chemistry,35,0.457143
physics,85,0.517647
writing-strategies,5,0.600000
world-history,6,0.666667
biology,99,0.696970
earth-science,37,0.756757
geography,48,0.791667
science-and-engineering-practices,48,0.895833
us-history,11,0.909091


In [ ]:
'''
# Muti-seed models
# Load One LoRA Checkpoint
def load_seed_model(seed):
    ckpt_dir = f"/content/multi_seed_models/seed_{seed}"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    base_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    seed_model = PeftModel.from_pretrained(base_model, ckpt_dir)
    seed_model.eval()
    return seed_model


# Load All Seed Models
seed_models = []

for seed in SEEDS:
    print("Loading seed model:", seed)
    seed_models.append(load_seed_model(seed))

print("Loaded models:", len(seed_models))
'''

In [ ]:
#  Create Submission CSV
def predict_test_answers(model, processor, test_df, img_root):
    predictions = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        pred, _ = score_choices_with_letters(model, processor, row, img_root)

        predictions.append({
            "id": row["id"],
            "answer": int(pred)
        })

    return pd.DataFrame(predictions)


# Run prediction on full test set
submission_df = predict_test_answers(
    model=model,
    processor=processor,
    test_df=test_df,
    img_root=IMG_ROOT
)

# Ensure correct format
submission_df["answer"] = submission_df["answer"].astype(int)

# Save file
SUBMISSION_PATH = "/content/submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print("Submission saved to:", SUBMISSION_PATH)
print(submission_df.head())

  0%|          | 0/1008 [00:00<?, ?it/s]

In [ ]:
'''
# Ensemble submission for muti-seed
submission_rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred, _ = ensemble_score_choices(seed_models, processor, row, IMG_ROOT)
    submission_rows.append({
        "id": row["id"],
        "answer": int(pred)
    })

submission_df = pd.DataFrame(submission_rows)
submission_df = submission_df[["id", "answer"]]
submission_df["answer"] = submission_df["answer"].astype(int)

SUBMISSION_PATH = "/content/submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(submission_df.head())
print("Saved:", SUBMISSION_PATH)
'''

In [ ]:
# Save Current Adapter
SAVE_DIR = f"/content/drive/MyDrive/smolvlm_lora_best{TRAIN_SUBSET_SIZE}_steps{MAX_TRAIN_STEPS}"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/smolvlm_lora_best"